## Import Libraries

In [1]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

C:\Users\imvik\AppData\Local\Temp\ipykernel_13764\3896410058.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


True

## Load and Chunk PDFs

In [2]:
faq_loader = PyPDFLoader("./PGRulesFAQ.pdf")
manual_loader = PyPDFLoader("./PG-Manual.pdf")

faq_docs = faq_loader.load()
manual_docs = manual_loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

faq_splits = text_splitter.split_documents(faq_docs)
manual_splits = text_splitter.split_documents(manual_docs)

print(f"FAQ chunks created: {len(faq_splits)}")
print(f"PG Manual chunks created: {len(manual_splits)}")

FAQ chunks created: 77
PG Manual chunks created: 347


## Embeddings and Vector Stores

In [3]:
# Using the same local HuggingFace embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

# Create two distinct Chroma databases in memory
faq_vectorstore = Chroma.from_documents(
    documents=faq_splits,
    embedding=embeddings,
    collection_name="faq_collection"
)

manual_vectorstore = Chroma.from_documents(
    documents=manual_splits,
    embedding=embeddings,
    collection_name="manual_collection"
)

# Create retrievers
faq_retriever = faq_vectorstore.as_retriever(search_kwargs={"k": 3})
manual_retriever = manual_vectorstore.as_retriever(search_kwargs={"k": 5})

print("Vector stores initialized successfully.")

Vector stores initialized successfully.


## LLM Pool and QA Chain Setup

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
import os

system_prompt = (
    "You are a highly analytical and precise academic research assistant. "
    "Use ONLY the following retrieved context to answer the question. "
    "Provide a comprehensive, well-structured answer that includes any relevant conditions, "
    "exceptions, or limits mentioned in the context. "
    "If the answer is not explicitly contained in the context, you must reply with exactly: "
    "'I do not know.' Do not guess, infer, or hallucinate.\n\n"
    "Context: {context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

# Build the Model Pool
chain_pool = []
gemini_keys = [os.getenv("GEMINI_API_KEY_1"), os.getenv("GEMINI_API_KEY_2")]
valid_g_keys = [k for k in gemini_keys if k]

gemini_models = ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-1.5-flash", "gemini-1.5-pro"]

for key in valid_g_keys:
    for m in gemini_models:
        llm = ChatGoogleGenerativeAI(model=m, google_api_key=key, temperature=0, max_retries=0)
        chain_pool.append({
            "name": f"Google: {m}",
            "llm": llm,
            "chain": create_stuff_documents_chain(llm, prompt)
        })
        
groq_key = os.getenv("GROQ_API_KEY")
if groq_key:
    groq_llm = ChatGroq(model="llama3-8b-8192", api_key=groq_key, temperature=0, max_retries=0)
    chain_pool.append({
        "name": "Groq: Llama 3 (8B)",
        "llm": groq_llm,
        "chain": create_stuff_documents_chain(groq_llm, prompt)
    })

print(f"Initialized LLM Pool with {len(chain_pool)} total models/keys for fallback.")

Initialized LLM Pool with 9 total models/keys for fallback.


## Hierarchical Search Logic with Explicit Model Tracking

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun
web_search_tool = DuckDuckGoSearchRun()
# Hyperlinks
FAQ_URL = "https://drive.google.com/file/d/1Sgsh7a_7_pKyr6YzKy2LiSiz-p96QEby/view?usp=sharing"
MANUAL_URL = "https://drive.google.com/file/d/1lmb6X882qgzsEzmZwCkrQMejKIiW4ynt/view?usp=sharing"

def expand_query(query):
    expansion_prompt = (
        "You are a strict query normalizer for IIT Kanpur academic policies. "
        "Standardize degree names and slang terms in the user query to match official IITK PG Manual terms. "
        "DO NOT add extra concepts. ONLY return the concise normalized query.\n\n"
        "Rules:\n"
        "- Standardize degrees: MTech/MT -> M.Tech. | PhD/phd -> Ph.D. | MSR/MS -> M.S. by Research\n"
        "- Keep 'MSPD' exactly as 'MSPD'.\n"
        "- Map slang: CG/CGPA/marks -> CPI | stipend/salary -> financial assistantship | guide -> thesis supervisor.\n\n"
        f"Original Query: {query}\n"
        "Normalized Query:"
    )
    # Loop through models until one succeeds
    for pool_item in chain_pool:
        try:
            return pool_item["llm"].invoke(expansion_prompt).content.strip()
        except:
            continue
    raise Exception("RESOURCE_EXHAUSTED")

def hierarchical_rag_search(query, use_web_search=False):
    try:
        # Guardrail: Handle Bot Self-Introduction Queries
        # Strip out punctuation so "Hi!" or "Who are you?" match perfectly
        clean_q = query.lower().strip().replace("?", "").replace("!", "").replace(".", "")
        
        intro_triggers = [
            "who are you", "what are you", "what can you do", 
            "what is this bot", "help", "help me", 
            "hi", "hello", "hey"
        ]
        
        # EXACT MATCH: Only trigger if the user's entire query is in the trigger list
        if clean_q in intro_triggers:
            return {
                "answer": (
                    "**Hello! I am the IITK PG Rules & FAQ Assistant.**\n\n"
                    "I am an AI assistant designed to help postgraduate students navigate academic policies at **IIT Kanpur**.\n\n"
                    "**What I can help you with:**\n"
                    "- **PG Manual Regulations:** Minimum CPI rules, assistantship/stipend eligibility, thesis submission timelines, course waiver conditions, and degree extensions.\n"
                    "- **FAQ Queries:** Common postgraduate administrative procedures and guidelines.\n"
                    "- **Web Search Fallback:** External official information regarding IIT Kanpur if rules aren't found in internal documents."
                ),
                "source_type": "System Bot Intro",
                "model_used": "Built-in Guardrail",
                "context_used": []
            }
   
        search_query = expand_query(query)
        print(f"-> Expanded Query: {search_query}")
        
        # Custom runner to execute the chain and capture the successful model name
        def run_qa_chain(context):
            for pool_item in chain_pool:
                try:
                    response = pool_item["chain"].invoke({"input": query, "context": context})
                    return response, pool_item["name"]
                except:
                    continue
            raise Exception("RESOURCE_EXHAUSTED")

        def run_web_llm(web_prompt):
            for pool_item in chain_pool:
                try:
                    response = pool_item["llm"].invoke(web_prompt).content
                    return response, pool_item["name"]
                except:
                    continue
            raise Exception("RESOURCE_EXHAUSTED")

        # Step A: FAQ
        faq_context = faq_retriever.invoke(search_query)
        faq_response, model_used = run_qa_chain(faq_context)
        
        if "i do not know" in faq_response.lower().strip().replace(".", ""):
            print("-> Answer not found in FAQ. Falling back to PG Manual...")
            
            # Step B: PG Manual
            manual_context = manual_retriever.invoke(search_query)
            manual_response, model_used = run_qa_chain(manual_context)
            
            if "i do not know" in manual_response.lower().strip().replace(".", ""):
                # Step C: Web
                if use_web_search:
                    print("-> Answer not found in Manual. Initiating Web Search...")
                    iitk_web_query = f"{query} IIT Kanpur" if "iit" not in query.lower() else query
                    web_results = web_search_tool.invoke(iitk_web_query)
                    web_prompt = f"Use these web results regarding IIT Kanpur to answer:\n\nQuery: {query}\n\nResults: {web_results}"
                    final_web_answer, model_used = run_web_llm(web_prompt)
                    
                    return {
                        "answer": f"**As your query was not found in the IITK FAQ or Manual, I searched the web:**\n\n{final_web_answer}",
                        "source_type": "Web Search",
                        "model_used": model_used,
                        "context_used": []
                    }
                else:
                    return {
                        "answer": "I do not know. The answer is not in the provided documents, and web search is disabled.",
                        "source_type": "None",
                        "model_used": model_used,
                        "context_used": []
                    }
                    
            return {"answer": manual_response, "source_type": "PG Manual", "model_used": model_used, "context_used": manual_context}
        
        return {"answer": faq_response, "source_type": "FAQ", "model_used": model_used, "context_used": faq_context}

    except Exception as e:
        if "RESOURCE_EXHAUSTED" in str(e):
            return {
                "answer": "**API Quota Exhausted!** \n\nAll primary and fallback LLMs have reached their daily rate limits. Please try again tomorrow.",
                "source_type": "System Error",
                "model_used": "N/A",
                "context_used": []
            }
        else:
            raise e

## Execute Test Query

In [6]:

query = "What is the minimum CPI required for an MSPD student to continue availing financial assistantship after the first semester?"
result = hierarchical_rag_search(query, use_web_search=True)

print(f"Answer: {result['answer']}\n")
print(f"Model Used: {result.get('model_used', 'N/A')}")

if result['source_type'] == "FAQ":
    print(f"Source Document: FAQ ({FAQ_URL})")
elif result['source_type'] == "PG Manual":
    print(f"Source Document: PG Manual ({MANUAL_URL})")
else:
    print(f"Source Document: {result['source_type']}")

if result['context_used']:
    print("\nCitations:")
    for i, doc in enumerate(result['context_used']):
        page_num = doc.metadata.get('page', 0) + 1
        print(f"[{i+1}] Page: {page_num}")

-> Expanded Query: What is the minimum CPI required for an MSPD student to continue availing financial assistantship after the first semester?
-> Answer not found in FAQ. Falling back to PG Manual...
Answer: After the first semester, MSPD students must have a minimum CPI of 6.5 to continue availing financial assistantship. If a student's CPI falls below 6.5, the assistantship will be discontinued until their CPI rises to 6.5 or above. The assistantship will then resume in the following semester, provided they have not migrated as a PhD student.

Model Used: Google: gemini-2.5-flash-lite
Source Document: PG Manual (https://drive.google.com/file/d/1lmb6X882qgzsEzmZwCkrQMejKIiW4ynt/view?usp=sharing)

Citations:
[1] Page: 16
[2] Page: 16
[3] Page: 48
[4] Page: 8
[5] Page: 12


## Automated RAG Evaluator

In [7]:
def evaluate_rag_response(query, context_docs, answer):
    """Evaluates Faithfulness and Relevance scores (0.0 to 1.0) for a RAG response."""
    
    context_text = "\n\n".join([doc.page_content for doc in context_docs]) if context_docs else "No context retrieved."
    
    eval_prompt = f"""
    You are an expert AI evaluator assessing a RAG system response.
    
    USER QUERY: {query}
    RETRIEVED CONTEXT: {context_text}
    GENERATED ANSWER: {answer}
    
    Evaluate the response on two metrics and assign a score from 0.0 to 1.0 for each:
    
    1. Faithfulness: Is every statement in the GENERATED ANSWER completely supported by the RETRIEVED CONTEXT? (1.0 = fully supported, 0.0 = total hallucination)
    2. Answer Relevance: Does the GENERATED ANSWER directly address the USER QUERY without unnecessary off-topic information? (1.0 = highly relevant, 0.0 = completely irrelevant)
    
    Respond STRICTLY in the following format:
    Faithfulness Score: <score>
    Relevance Score: <score>
    Reasoning: <one brief sentence explaining the scores>
    """
    
    # Use the robust chain_pool instead of a single global LLM
    for pool_item in chain_pool:
        try:
            eval_result = pool_item["llm"].invoke(eval_prompt).content.strip()
            return f"{eval_result}\n(Evaluator Model: {pool_item['name']})"
        except Exception as e:
            continue
            
    return "Evaluation Failed: All models in the pool are exhausted or unavailable."

In [8]:
# Example evaluation
query = "When can I submit my Ph.D. thesis?"
result = hierarchical_rag_search(query, use_web_search=False)

print(f"Answer: {result['answer']}\n")

# Run Evaluation if answer came from documents
if result["context_used"]:
    print("\n--- RAG EVALUATION REPORT ---")
    score_report = evaluate_rag_response(query, result["context_used"], result["answer"])
    print(score_report)

-> Expanded Query: When can I submit my Ph.D. thesis?
Answer: You can submit your Ph.D. thesis only after the thesis board has been duly approved and you have received intimation of the satisfactory completion of the open seminar. This is further qualified by the condition that the thesis board may be proposed at the most 4 weeks before the submission of the thesis. Additionally, you can submit your thesis only after 4 weeks of approval of your thesis board and the intimation of the satisfactory completion of the open seminar has been received.


--- RAG EVALUATION REPORT ---
Faithfulness Score: 1.0
Relevance Score: 1.0
Reasoning: The generated answer accurately synthesizes information from the provided context to directly answer the user's question about thesis submission timelines.
(Evaluator Model: Google: gemini-2.5-flash-lite)


In [9]:
# Example evaluation
query = "Help"
result = hierarchical_rag_search(query, use_web_search=False)

print(f"Answer: {result['answer']}\n")

Answer: **Hello! I am the IITK PG Rules & FAQ Assistant.**

I am an AI assistant designed to help postgraduate students navigate academic policies at **IIT Kanpur**.

**What I can help you with:**
- **PG Manual Regulations:** Minimum CPI rules, assistantship/stipend eligibility, thesis submission timelines, course waiver conditions, and degree extensions.
- **FAQ Queries:** Common postgraduate administrative procedures and guidelines.
- **Web Search Fallback:** External official information regarding IIT Kanpur if rules aren't found in internal documents.



## Integrity & Routing Test Suite

In [ ]:
# import time

# test_suite = [
#     {"query": "When can I submit my Ph.D. thesis?", "expected_source": "FAQ"},
#     {"query": "What is the minimum CPI required for an MSPD student to continue availing financial assistantship after the first semester?", "expected_source": "PG Manual"},
#     {"query": "What is the capital of France?", "expected_source": "Web Search"}
# ]

# print("Starting Automated Pipeline Integrity Test...\n")

# for idx, test in enumerate(test_suite):
#     print(f"--- TEST CASE {idx + 1} ---")
#     print(f"Query: {test['query']}")
    
#     result = hierarchical_rag_search(test['query'], use_web_search=True)
    
#     routing_passed = result['source_type'] == test['expected_source']
#     status = "PASSED" if routing_passed else "FAILED"
    
#     print(f"Expected Source: {test['expected_source']} | Detected Source: {result['source_type']}")
#     print(f"Status: {status}")
#     print(f"Answer:\n{result['answer']}\n")
#     print("-" * 60)
    
#     # Pause for 10 seconds before the next loop to respect the free tier RPM limits
#     if idx < len(test_suite) - 1:
#         print("Pausing for 3 seconds to respect API rate limits...\n")
#         time.sleep(3)